# Lab 10A Lakehouse Federation

This notebook queries the existing PostgreSQL foreign catalog and compares it with the existing Delta snapshot. It contains no database credentials.

- Catalog: `lab10_postgres_catalog`
- Connection: `lab10_postgres`
- Database: `lab10db`
- Schema: `public`

## Query the foreign PostgreSQL customers table

In [ ]:
%sql
SELECT customer_id, customer_name, country, segment
FROM lab10_postgres_catalog.public.customers
ORDER BY customer_id

Observed result (**5 rows**):

| customer_id | customer_name | country | segment |
|---:|---|---|---|
| 1 | Alice Johnson | Poland | Consumer |
| 2 | Mark Smith | Germany | Corporate |
| 3 | Sofia Brown | Poland | Consumer |
| 4 | Daniel Wilson | France | Corporate |
| 5 | Emma Davis | Germany | Consumer |

## Query the foreign PostgreSQL orders table

In [ ]:
%sql
SELECT order_id, customer_id, order_date, amount, status
FROM lab10_postgres_catalog.public.orders
ORDER BY order_id

Observed result (**5 rows**):

| order_id | customer_id | order_date | amount | status |
|---:|---:|---|---:|---|
| 1001 | 1 | 2026-09-01 | 250.00 | Completed |
| 1002 | 2 | 2026-09-02 | 800.00 | Completed |
| 1003 | 3 | 2026-09-03 | 150.00 | Pending |
| 1004 | 4 | 2026-09-04 | 1200.00 | Completed |
| 1005 | 5 | 2026-09-05 | 450.00 | Cancelled |

## Compare the Delta copy

In [ ]:
%sql
SELECT customer_id, customer_name, country, segment
FROM dbr_dev.ayyuborujzade_bronze.lab10_customers_delta
ORDER BY customer_id

The Delta copy contained **5 rows**, matching the foreign customers table at validation time.

## Federation-to-Delta join

In [ ]:
%sql
SELECT p.customer_id AS postgresql_customer_id,
       p.customer_name AS postgresql_customer_name,
       p.country,
       p.segment,
       d.customer_name AS delta_customer_name
FROM lab10_postgres_catalog.public.customers AS p
INNER JOIN dbr_dev.ayyuborujzade_bronze.lab10_customers_delta AS d
  ON p.customer_id = d.customer_id
ORDER BY p.customer_id

Observed join result (**5 matching rows**):

| PostgreSQL ID | PostgreSQL customer | Country | Segment | Delta customer |
|---:|---|---|---|---|
| 1 | Alice Johnson | Poland | Consumer | Alice Johnson |
| 2 | Mark Smith | Germany | Corporate | Mark Smith |
| 3 | Sofia Brown | Poland | Consumer | Sofia Brown |
| 4 | Daniel Wilson | France | Corporate | Daniel Wilson |
| 5 | Emma Davis | Germany | Consumer | Emma Davis |

## Federation and ingestion

Federation queries remote tables through Unity Catalog without first loading them into Delta. The query sees current source data, but network and remote database response times add latency. A Delta copy is ingested data: it can be stale until refreshed, while repeated analytics can use local storage and compute.

Keep credentials in the workspace-managed connection, grant the connection and foreign catalog only to intended identities, and use the least database privileges needed. Do not put credentials in notebooks or source control.